# 05a-mm-flip-b2: combined model with EfficientNet-B2

Same as b0-v1-mm-flip except the backbone (EfficientNet-B2 instead of B0).
Compare with b0-v1-mm-flip to see what a bigger backbone adds.

Inputs: competition data, outputs of 01 (study_inventory.csv), 04 (silver_scores.csv),
02d shard 0 and shard 1 (cache_mm), 05c (laterality_final.csv), utility script knee_model.
Settings: GPU T4 x2, Internet on (pretrained weights, W&B).

# Cell 1: inputs, folds, config

In [ ]:
import os, sys, glob, json, time, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

SANITY = False
EXP = "b2-v1-mm-flip" + ("-sanity" if SANITY else "")
FOLDS = [0, 1] if SANITY else [0, 1, 2, 3, 4]
ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT = Path("/kaggle/working")

def find_all(pattern):
    hits = glob.glob(f"/kaggle/usr/lib/**/{pattern}", recursive=True)
    for top in os.listdir("/kaggle/input"):
        if top != "competitions":
            hits += glob.glob(f"/kaggle/input/{top}/**/{pattern}", recursive=True)
    return sorted(hits)

def find_one(pattern):
    hits = find_all(pattern)
    print(f"{pattern}: {hits[:3]}")
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]

KM_DIR = os.path.dirname(find_one("knee_model.py"))
LABELS_CSV = find_one("silver_scores.csv")
CACHE_DIR = [d for d in find_all("cache_mm") if os.path.isdir(d)]
INV_CSV = find_one("study_inventory.csv")
LAT_CSV = find_one("laterality_final.csv")
print("cache:", CACHE_DIR, "npz files:", sum(len(glob.glob(f"{d}/*.npz")) for d in CACHE_DIR))

sys.path.insert(0, KM_DIR)
import knee_model as km

silver = pd.read_csv(LABELS_CSV)
inv = pd.read_csv(INV_CSV)
df = silver[["StudyInstanceUID"]].merge(inv[["StudyInstanceUID", "lang", "vendor", "is_gold"]],
                                        on="StudyInstanceUID", how="left")
df["site"] = df["lang"].fillna("na") + "_" + df["vendor"].fillna("na")
df["fold"] = -1
for f, (_, va) in enumerate(GroupKFold(n_splits=5).split(df, groups=df["site"])):
    df.loc[va, "fold"] = f
print(df.groupby("fold").agg(studies=("site", "size"), sites=("site", "nunique"),
                             gold=("is_gold", "sum")).to_string())
df[["StudyInstanceUID", "fold", "site"]].to_csv(OUT / "folds.csv", index=False)

CFG = dict(exp=EXP, backbone="efficientnet_b2", epochs=2 if SANITY else 8, bs=2, accum=4, lr=3e-4, wd=1e-2,
           drop=0.3, workers=2, seed=42, wandb_project="knee-mri-abnormality",
           labels_csv=LABELS_CSV, cache_dir=CACHE_DIR, geometry="mm", flip=True, lat_csv=LAT_CSV, folds_csv=str(OUT / "folds.csv"), out_dir=str(OUT))
print(json.dumps(CFG, indent=1))

# Cell 2: environment, pretrained weights, W&B

In [ ]:
import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__,
      "|", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
timm.create_model(CFG["backbone"], pretrained=True)
print("pretrained weights cached")

try:
    from kaggle_secrets import UserSecretsClient
    import wandb
    key = UserSecretsClient().get_secret("WANDB_API_KEY")
    wandb.login(key=key, relogin=True, verify=True)
    os.environ["WANDB_API_KEY"] = key
    print("W&B login ok, wandb", wandb.__version__)
except Exception as err:
    print("no W&B:", repr(err)[:200])

print("new W&B block attached:", "wandb run:" in open(f"{KM_DIR}/knee_model.py").read())

# Cell 3: train folds, one per GPU

In [ ]:
def launch(fold, gpu):
    cfg_path = OUT / f"cfg_f{fold}.json"
    json.dump({**CFG, "fold": fold}, open(cfg_path, "w"), indent=1)
    code = (f"import sys, json; sys.path.insert(0, {KM_DIR!r}); import knee_model as km; "
            f"km.train_fold(json.load(open({str(cfg_path)!r})))")
    log = open(OUT / f"log_f{fold}.txt", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    return subprocess.Popen([sys.executable, "-c", code], env=env, stdout=log, stderr=subprocess.STDOUT)

def last_line(fold):
    lines = [l for l in open(OUT / f"log_f{fold}.txt").read().splitlines() if l.strip()]
    return lines[-1][:160] if lines else ""

t0 = time.time()
queue, running, free, seen = list(FOLDS), {}, list(range(torch.cuda.device_count())), {}
while queue or running:
    while queue and free:
        g, f = free.pop(0), queue.pop(0)
        running[f] = (launch(f, g), g)
        print(f"started fold {f} on gpu {g}")
    time.sleep(30)
    for f, (p, g) in list(running.items()):
        line = last_line(f)
        if line != seen.get(f):
            print(f"[{(time.time() - t0) / 60:5.1f} min] fold {f}: {line}")
            seen[f] = line
        if p.poll() is not None:
            print(f"fold {f} exited with code {p.returncode}")
            if p.returncode != 0:
                print(open(OUT / f"log_f{f}.txt").read()[-2000:])
            free.append(g)
            del running[f]
print(f"all folds done in {(time.time() - t0) / 60:.1f} min")

# Cell 4: out-of-fold scores (silver and gold)

In [ ]:
from sklearn.metrics import roc_auc_score

oof_files = sorted(OUT.glob("oof_f*.csv"))
if not oof_files:
    raise SystemExit("no out-of-fold files: check the fold logs in cell 3")
oof = pd.concat([pd.read_csv(p) for p in oof_files], ignore_index=True)
print("oof studies:", len(oof))

sil = silver.merge(oof, on="StudyInstanceUID", suffixes=("", "_p"))
s_auc = {}
for k in km.LABELS:
    y = (sil[k] >= km.SILVER_POS).astype(int)
    if 0 < y.sum() < len(y):
        s_auc[k] = roc_auc_score(y, sil[k + "_p"])

train = pd.read_csv(f"{ROOT}/train.csv")
gold = train[train[km.LABELS].notna().all(axis=1)][["StudyInstanceUID"] + km.LABELS]
gg = gold.merge(oof, on="StudyInstanceUID", suffixes=("", "_p"))
g_auc = {k: roc_auc_score(gg[k], gg[k + "_p"]) for k in km.LABELS if gg[k].nunique() == 2}

table = pd.DataFrame({"silver_oof": s_auc, "gold_oof": g_auc}).round(3)
print(table.to_string())
print("macro:", table.mean().round(3).to_dict(), "| gold studies in oof:", len(gg))
json.dump({"exp": EXP, "silver_oof": s_auc, "gold_oof": g_auc, "gold_n": int(len(gg))},
          open(OUT / "oof_summary.json", "w"), indent=1)